# 03 - DDXPlus Feature Engineering

This notebook builds the Healix feature pipeline directly from the DDXPlus raw split in `app/data/raw/ddxplus/train.csv` and turns it into a binary evidence matrix compatible with the downstream ML workflow.

In [1]:
from pathlib import Path
import ast
import json

import pandas as pd
import numpy as np
import joblib
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, MultiLabelBinarizer
# Ensure repository root is on sys.path so local package imports work in the notebook
repo_root = Path(r"c:\healix-ai-medical-assistant")
import sys
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))
from app.domain.clinical_record import ClinicalRecord
# Load the raw DDXPlus split
raw_path = repo_root / "app" / "data" / "raw" / "ddxplus" / "train.csv"

if not raw_path.exists():
    raise FileNotFoundError(f"Raw dataset not found: {raw_path}")

df = pd.read_csv(raw_path, usecols=["AGE", "DIFFERENTIAL_DIAGNOSIS", "SEX", "PATHOLOGY", "EVIDENCES", "INITIAL_EVIDENCE"])
print("Loaded raw DDXPlus split:", raw_path)
print("Rows:", len(df))
print("Columns:", list(df.columns))


def split_token(token: str) -> str:
    return str(token).split("_@_", 1)[0]


def parse_evidences(value):
    text = "[]" if pd.isna(value) else str(value)
    try:
        tokens = ast.literal_eval(text)
    except (ValueError, SyntaxError):
        tokens = []
    return [split_token(token) for token in tokens]


df["evidence_tokens"] = df["EVIDENCES"].apply(parse_evidences)
df["AGE"] = pd.to_numeric(df["AGE"], errors="coerce")
df["SEX_M"] = df["SEX"].apply(lambda value: 1 if str(value).strip().upper() == "M" else 0)

all_evidence_codes = sorted({code for tokens in df["evidence_tokens"] for code in tokens})
mlb = MultiLabelBinarizer(classes=all_evidence_codes)
evidence_matrix = mlb.fit_transform(df["evidence_tokens"]).astype(np.uint8)
# Raw DDXPlus tokens already carry the E_ prefix (e.g. "E_53") -- do not prefix again.
evidence_columns = [str(code) for code in mlb.classes_]
evidence_df = pd.DataFrame(evidence_matrix, columns=evidence_columns, index=df.index)

print("Evidence features created:", len(evidence_columns))
print("Sample evidence columns:", evidence_columns[:10])

Loaded raw DDXPlus split: c:\healix-ai-medical-assistant\app\data\raw\ddxplus\train.csv
Rows: 1025602
Columns: ['AGE', 'DIFFERENTIAL_DIAGNOSIS', 'SEX', 'PATHOLOGY', 'EVIDENCES', 'INITIAL_EVIDENCE']


Evidence features created: 223
Sample evidence columns: ['E_0', 'E_1', 'E_10', 'E_100', 'E_101', 'E_102', 'E_103', 'E_104', 'E_105', 'E_106']


In [2]:
# A Healix-style ClinicalRecord example built from a real DDXPlus row
sample_row = df.iloc[0]
clinical_record = ClinicalRecord(
    chief_complaint=str(sample_row["INITIAL_EVIDENCE"]),
    severity="moderate",
    duration="unknown",
    body_location="unknown",
    medications=[],
    allergies=[],
    chronic_conditions=[],
    family_history=[],
    missing_fields=[],
)
print(clinical_record)
print("Evidence tokens from the same row:", sample_row["evidence_tokens"][:10])

ClinicalRecord(chief_complaint='E_91', severity='moderate', duration='unknown', body_location='unknown', medications=[], allergies=[], chronic_conditions=[], family_history=[], missing_fields=[])
Evidence tokens from the same row: ['E_48', 'E_50', 'E_53', 'E_54', 'E_54', 'E_55', 'E_55', 'E_55', 'E_56', 'E_57']


In [3]:
# Build the DDXPlus feature vocabulary from the raw split
feature_columns = ["AGE", "SEX_M"] + evidence_columns
print("Feature vector length:", len(feature_columns))
print("First 20 features:", feature_columns[:20])

Feature vector length: 225
First 20 features: ['AGE', 'SEX_M', 'E_0', 'E_1', 'E_10', 'E_100', 'E_101', 'E_102', 'E_103', 'E_104', 'E_105', 'E_106', 'E_107', 'E_108', 'E_109', 'E_11', 'E_110', 'E_111', 'E_112', 'E_113']


In [4]:
# Implement the mapping: ClinicalRecord + evidence tokens -> DDXPlus-style feature vector

def clinical_record_to_features(record, evidence_tokens, feature_names):
    feature_vector = {name: 0 for name in feature_names}
    feature_vector["AGE"] = getattr(record, "age", None) if hasattr(record, "age") else None
    feature_vector["SEX_M"] = 1 if str(getattr(record, "sex", "M")).strip().upper() == "M" else 0

    present = {split_token(token) for token in evidence_tokens if str(token).strip()}
    for code in present:
        # code already carries the E_ prefix from split_token -- do not prefix again.
        if code in feature_vector:
            feature_vector[code] = 1

    return feature_vector

mapped = clinical_record_to_features(clinical_record, sample_row["evidence_tokens"], feature_columns)
print("Mapped feature vector sample:")
print({k: mapped[k] for k in feature_columns[:12]})

Mapped feature vector sample:
{'AGE': None, 'SEX_M': 1, 'E_0': 0, 'E_1': 0, 'E_10': 0, 'E_100': 0, 'E_101': 0, 'E_102': 0, 'E_103': 0, 'E_104': 0, 'E_105': 0, 'E_106': 0}


## Mapping strategy

- The raw DDXPlus evidence grammar is parsed into base evidence codes.
- Each code becomes a binary feature such as `E_55` or `E_204`.
- Missing evidence tokens are simply left at `0`; no fabricated symptoms are added.
- `AGE` and `SEX_M` are kept as metadata features alongside the evidence vector.
- This creates a deterministic feature space that can be reused during training and runtime inference.

In [5]:
# Create train/test split from the raw DDXPlus-derived feature matrix
label_encoder = LabelEncoder()
df["target"] = label_encoder.fit_transform(df["PATHOLOGY"])
X = pd.concat([df[["AGE", "SEX_M"]].reset_index(drop=True), evidence_df.reset_index(drop=True)], axis=1)
y = df["target"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

X_train shape: (769201, 225)
X_test shape: (256401, 225)
y_train shape: (769201,)
y_test shape: (256401,)


In [6]:
# Save the engineered datasets and feature names
out_dir = repo_root / "training" / "data" / "processed"
out_dir.mkdir(parents=True, exist_ok=True)
train_path = out_dir / "X_train.csv"
test_path = out_dir / "X_test.csv"
train_labels_path = out_dir / "y_train.csv"
test_labels_path = out_dir / "y_test.csv"

X_train.to_csv(train_path, index=False)
X_test.to_csv(test_path, index=False)
y_train.to_frame(name="target").to_csv(train_labels_path, index=False)
y_test.to_frame(name="target").to_csv(test_labels_path, index=False)

print("Saved train/test matrices:")
print(train_path)
print(test_path)

Saved train/test matrices:
c:\healix-ai-medical-assistant\training\data\processed\X_train.csv
c:\healix-ai-medical-assistant\training\data\processed\X_test.csv


In [7]:
# Save the feature names in the exact order used by the runtime encoder
training_models_dir = repo_root / "training" / "models"
training_models_dir.mkdir(parents=True, exist_ok=True)
runtime_models_dir = repo_root / "models"
runtime_models_dir.mkdir(parents=True, exist_ok=True)
feature_names = list(feature_columns)
(training_models_dir / "feature_names.json").write_text(json.dumps(feature_names, indent=2), encoding="utf-8")
(runtime_models_dir / "feature_names.json").write_text(json.dumps(feature_names, indent=2), encoding="utf-8")
joblib.dump(label_encoder, training_models_dir / "label_encoder.joblib")
joblib.dump(label_encoder, runtime_models_dir / "label_encoder.joblib")
print("Feature names saved for export")

Feature names saved for export
